# Smol LM - Tiny Transformer

A small decoder-only transformer with:
- **RoPE** (Rotary Position Embeddings)
- **GQA** (Grouped Query Attention)
- **RMSNorm**
- **SWiGLU** FFN
- **Weight tying** (token embedding & lm head)

Trains on Shakespeare in ~5 minutes on Colab's free T4 GPU.

In [0]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from typing import List, Tuple
import time
import urllib.request

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load & tokenize Shakespeare

In [0]:
# Download Shakespeare if not already present
import os
if not os.path.exists("shakespeare.txt"):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "shakespeare.txt")
    print("Downloaded shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

print(f"Text length: {len(text):,} chars")
print(f"Vocab size: {vocab_size} unique chars")

def encode(s: str) -> List[int]:
    return [char_to_idx[c] for c in s]

def decode(vec: List[int]) -> str:
    return "".join([idx_to_char[i] for i in vec])

In [0]:
cut = int(len(text) * 0.9)
train_data = encode(text[:cut])
val_data = encode(text[cut:])
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")

def get_batch(split: str, batch_size: int, context_length: int):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - context_length, (batch_size,))
    x = torch.stack([torch.tensor(d[i : i + context_length]) for i in ix.tolist()])
    y = torch.stack([torch.tensor(d[i + 1 : i + context_length + 1]) for i in ix.tolist()])
    return x.to(device), y.to(device)

## 2. Model Architecture

In [0]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

In [0]:
def precompute_rope_freqs(
    head_dim: int, max_seq_len: int, base: float = 10000.0
) -> Tuple[Tensor, Tensor]:
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, freqs)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
    seq_len = x.shape[2]
    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    out1 = x1 * cos - x2 * sin
    out2 = x1 * sin + x2 * cos
    return torch.stack([out1, out2], dim=-1).flatten(-2)

In [0]:
def repeat_kv(x: Tensor, n_rep: int) -> Tensor:
    if n_rep == 1:
        return x
    b, n_kv, seq, hd = x.shape
    return (
        x[:, :, None, :, :]
        .expand(b, n_kv, n_rep, seq, hd)
        .reshape(b, n_kv * n_rep, seq, hd)
    )


class GQA(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        head_dim: int,
        n_kv_heads: int,
        max_seq_len: int,
    ) -> None:
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(d_model, n_heads * head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim)
        self.o_proj = nn.Linear(n_heads * head_dim, d_model)
        self.rope_cos, self.rope_sin = precompute_rope_freqs(head_dim, max_seq_len)

    def forward(self, x: Tensor) -> Tensor:
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        b, seq, _ = x.shape
        q = q.view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, seq, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, seq, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)
        scale = 1.0 / math.sqrt(self.head_dim)
        scores = (q @ k.transpose(-2, -1)) * scale
        mask = torch.triu(torch.ones(seq, seq, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = F.dropout(weights, p=0.2, training=self.training)
        out = weights @ v
        out = out.transpose(1, 2).contiguous()
        out = out.view(b, seq, self.n_heads * self.head_dim)
        return self.o_proj(out)

In [0]:
class SWiGLU(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int) -> None:
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden_dim)
        self.w_up = nn.Linear(d_model, hidden_dim)
        self.w_down = nn.Linear(hidden_dim, d_model)

    def forward(self, x: Tensor):
        gate = F.silu(self.w_gate(x))
        up = self.w_up(x)
        return F.dropout(self.w_down(gate * up), p=0.2, training=self.training)

In [0]:
class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_kv_heads: int,
        n_heads: int,
        ffn_hidden_dims: int,
        max_seq_len: int,
    ) -> None:
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.ffn_norm = RMSNorm(d_model)
        head_dim = d_model // n_heads
        self.attention = GQA(d_model, n_heads, head_dim, n_kv_heads, max_seq_len)
        self.ffn = SWiGLU(d_model, ffn_hidden_dims)

    def forward(self, x: Tensor):
        x = x + self.attention(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x

In [0]:
class Smol_LM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        n_heads: int = 8,
        n_kv_heads: int = 2,
        num_layers: int = 6,
        ffn_hidden_mult: int = 4,
        max_seq_len: int = 256,
    ) -> None:
        super().__init__()
        self.max_seq_len = max_seq_len
        head_dim = d_model // n_heads
        ffn_hidden_dims = ffn_hidden_mult * d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(
            [
                TransformerBlock(d_model, n_kv_heads, n_heads, ffn_hidden_dims, max_seq_len)
                for _ in range(num_layers)
            ]
        )
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.token_emb.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x: Tensor, targets: Tensor | None = None) -> Tuple[Tensor, Tensor | None]:
        x = self.token_emb(x)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx: Tensor, max_new_tokens: int, temperature: float = 1.0) -> Tensor:
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len :]
            logits, _ = self.forward(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 3. Training

In [0]:
d_model = 256
n_heads = 8
n_kv_heads = 2
num_layers = 6
max_seq_len = 256
batch_size = 64
learning_rate = 3e-4
max_iters = 5000
eval_interval = 500
eval_iters = 20

model = Smol_LM(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_kv_heads=n_kv_heads,
    num_layers=num_layers,
    max_seq_len=max_seq_len,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {total_params:,}")
print(f"Context length: {max_seq_len}")
print(f"Batch size: {batch_size}")

In [0]:
@torch.no_grad()
def evaluate(model, batch_size, context_length, num_batches=20):
    model.eval()
    total_loss = 0.0
    for _ in range(num_batches):
        x, y = get_batch("val", batch_size, context_length)
        _, loss = model(x, y)
        total_loss += loss.item()
    model.train()
    return total_loss / num_batches


optimizer = AdamW(model.parameters(), lr=learning_rate)
scheduler = CosineAnnealingLR(optimizer, T_max=max_iters)

start_time = time.time()
best_val_loss = float("inf")

for step in range(max_iters + 1):
    x, y = get_batch("train", batch_size, max_seq_len)
    optimizer.zero_grad()
    _, loss = model(x, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if step % eval_interval == 0:
        val_loss = evaluate(model, batch_size, max_seq_len, eval_iters)
        elapsed = time.time() - start_time
        print(f"Step {step:5d} | train loss {loss.item():.4f} | val loss {val_loss:.4f} | elapsed {elapsed:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "smol_lm_best.pt")
            print(f"  -> saved best model (val_loss={val_loss:.4f})")

        if step > 0:
            context = torch.tensor([encode("ROMEO: ")], dtype=torch.long, device=device)
            output = model.generate(context, max_new_tokens=200, temperature=0.8)
            generated = decode(output[0].tolist())
            print(f"  Sample:\n{generated}\n")
            print("-" * 60)

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time:.1f}s")
print(f"Best val loss: {best_val_loss:.4f}")

## 4. Generate text from the trained model

In [0]:
# Load best model
model.load_state_dict(torch.load("smol_lm_best.pt", map_location=device))
model.eval()

prompt = "JULIET: "
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
output = model.generate(context, max_new_tokens=500, temperature=0.7)
print(decode(output[0].tolist()))